In [ ]:
batch_id = "manual-dev-run"
evaluation_path = "Files/_evaluation"

In [ ]:
%run 00_silver_common

In [ ]:
truth = (
    spark.read.option("header", "true")
    .csv(f"{evaluation_path}/_mdm_truth.csv")
    .filter((F.col("dms_customer_id") != "") & (F.col("crm_id") != ""))
    .select(
        F.col("dms_customer_id").alias("truth_dms"),
        F.col("crm_id").alias("truth_crm"),
        F.col("match_difficulty"),
        F.col("customer_type"),
    )
)

In [ ]:
predicted = (
    spark.table("silver.customer_xref")
    .filter(F.col("dms_customer_id").isNotNull() & F.col("crm_id").isNotNull())
    .select(
        F.col("dms_customer_id").alias("pred_dms"),
        F.col("crm_id").alias("pred_crm"),
        F.col("match_score"),
    )
)

In [ ]:
truth.cache()

In [ ]:
predicted.cache()

In [ ]:
truth_pairs = truth.count()

In [ ]:
predicted_pairs = predicted.count()

In [ ]:
scored = (
    truth.join(
        predicted,
        (F.col("truth_dms") == F.col("pred_dms")) & (F.col("truth_crm") == F.col("pred_crm")),
        how="full_outer",
    )
    .withColumn(
        "classification",
        F.when(F.col("truth_dms").isNotNull() & F.col("pred_dms").isNotNull(), "TP")
        .when(F.col("pred_dms").isNotNull(), "FP")
        .otherwise("FN"),
    )
)

In [ ]:
counts = {
    row["classification"]: row["n"]
    for row in scored.groupBy("classification").agg(F.count("*").alias("n")).collect()
}

In [ ]:
tp, fp, fn = counts.get("TP", 0), counts.get("FP", 0), counts.get("FN", 0)

In [ ]:
precision = tp / (tp + fp) if tp + fp else 0.0

In [ ]:
recall = tp / (tp + fn) if tp + fn else 0.0

In [ ]:
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

In [ ]:
print("MDM MATCHING")

In [ ]:
print(f"  matchable pairs in truth : {truth_pairs:,}")

In [ ]:
print(f"  pairs the platform made  : {predicted_pairs:,}")

In [ ]:
print(f"  true positives           : {tp:,}")

In [ ]:
print(f"  false positives          : {fp:,}   <- wrong merges, the expensive kind")

In [ ]:
print(f"  false negatives          : {fn:,}   <- missed merges, the visible kind")

In [ ]:
print(f"  precision                : {precision:.2%}")

In [ ]:
print(f"  recall                   : {recall:.2%}")

In [ ]:
print(f"  F1                       : {f1:.2%}")

In [ ]:
by_difficulty = (
    truth.join(
        predicted,
        (F.col("truth_dms") == F.col("pred_dms")) & (F.col("truth_crm") == F.col("pred_crm")),
        how="left",
    )
    .groupBy("match_difficulty")
    .agg(
        F.count("*").alias("pairs_in_truth"),
        F.sum(F.when(F.col("pred_dms").isNotNull(), 1).otherwise(0)).alias("found"),
    )
    .withColumn("recall", F.round(F.col("found") / F.col("pairs_in_truth"), 4))
    .orderBy("match_difficulty")
)

In [ ]:
by_difficulty.show(truncate=False)

In [ ]:
by_type = (
    truth.join(
        predicted,
        (F.col("truth_dms") == F.col("pred_dms")) & (F.col("truth_crm") == F.col("pred_crm")),
        how="left",
    )
    .groupBy("customer_type")
    .agg(
        F.count("*").alias("pairs_in_truth"),
        F.sum(F.when(F.col("pred_dms").isNotNull(), 1).otherwise(0)).alias("found"),
    )
    .withColumn("recall", F.round(F.col("found") / F.col("pairs_in_truth"), 4))
)

In [ ]:
by_type.show(truncate=False)

In [ ]:
missed = (
    truth.join(
        predicted,
        (F.col("truth_dms") == F.col("pred_dms")) & (F.col("truth_crm") == F.col("pred_crm")),
        how="left_anti",
    )
    .limit(25)
)

In [ ]:
dms_names = spark.table("bronze.dms_customer").select(
    F.col("customer_id").alias("truth_dms"),
    F.col("full_name").alias("dms_name"),
    F.col("identity_no").alias("dms_identity"),
)

In [ ]:
crm_names = spark.table("bronze.crm_customer").select(
    F.col("crm_id").alias("truth_crm"),
    F.col("full_name").alias("crm_name"),
    F.col("identity_no").alias("crm_identity"),
)

In [ ]:
print("a sample of pairs the matcher did not find:")

In [ ]:
(
    missed.join(dms_names, on="truth_dms", how="left")
    .join(crm_names, on="truth_crm", how="left")
    .select("match_difficulty", "dms_name", "crm_name", "dms_identity", "crm_identity")
    .show(25, truncate=False)
)

In [ ]:
injected = (
    spark.read.option("header", "true")
    .csv(f"{evaluation_path}/_injection_log.csv")
    .select("rule_code", "business_key", "target_entity", "column_name", "cascade_rules")
)

In [ ]:
detected = spark.table("silver.dq_violation").filter(
    F.col("batch_id") == batch_id
).select("rule_code", "business_key")

In [ ]:
dq = (
    injected.alias("i")
    .join(
        detected.alias("d"),
        (F.col("i.rule_code") == F.col("d.rule_code"))
        & (F.col("i.business_key") == F.col("d.business_key")),
        how="left",
    )
    .groupBy(F.col("i.rule_code").alias("rule_code"))
    .agg(
        F.count("*").alias("injected"),
        F.sum(F.when(F.col("d.business_key").isNotNull(), 1).otherwise(0)).alias("detected"),
    )
    .withColumn("recall", F.round(F.col("detected") / F.col("injected"), 4))
    .orderBy(F.col("recall").asc())
)

In [ ]:
print("DQ RECALL BY RULE  (lowest first - the interesting end)")

In [ ]:
dq.show(30, truncate=False)

In [ ]:
totals = dq.agg(
    F.sum("injected").alias("injected"), F.sum("detected").alias("detected")
).collect()[0]

In [ ]:
print(
    f"overall DQ recall: {totals['detected']:,} / {totals['injected']:,} = "
    f"{totals['detected'] / totals['injected']:.2%}"
)

In [ ]:
unexplained = (
    detected.alias("d")
    .join(
        injected.alias("i"),
        (F.col("i.rule_code") == F.col("d.rule_code"))
        & (F.col("i.business_key") == F.col("d.business_key")),
        how="left_anti",
    )
    .groupBy("rule_code")
    .agg(F.count("*").alias("detected_without_injection"))
    .orderBy(F.col("detected_without_injection").desc())
)

In [ ]:
unexplained.show(30, truncate=False)

In [ ]:
print(
    "\nRecord these figures in docs/interview_defense_notes.md. "
    "A measured number with its failure modes explained is worth more than a "
    "round number with none."
)